## Imports

In [10]:
import pandas as pd
import numpy as np
import scipy
import seaborn as sns
import matplotlib.pyplot as plt

### Leitura do CSV baixado do Kaggle

In [11]:
caminho_arquivo = "AI_Impact_Student_Life_2026.csv"

df_completo = pd.read_csv(
    caminho_arquivo,
    sep=",",
    encoding="utf-8",
    decimal="."
)

df_completo.columns = df_completo.columns.str.strip()

### Seleção apenas das colunas de interesse

In [12]:
colunas_interesse = [
    "Student_ID",
    "Task_Frequency_Daily",
    "Main_Usage_Case",
    "GPA_Baseline",
    "GPA_Post_AI"
]

df = df_completo[colunas_interesse].copy()

### Inspeção inicial

In [ ]:
print("Formato do DataFrame (linhas, colunas):", df.shape)
print("\nPrimeiras linhas:")
print(df.head())

print("\nTipos de dados por coluna:")
print(df.dtypes)

print("\nValores ausentes por coluna:")
print(df.isnull().sum())

print("\nResumo estatístico (colunas numéricas):")
print(df.describe())

# ---- Identificação de dados duplicados ----

# Duplicatas exatas (todas as colunas iguais)
qtd_duplicadas_exatas = df.duplicated().sum()
print(f"\nQuantidade de linhas totalmente duplicadas: {qtd_duplicadas_exatas}")

# Duplicatas por Student_ID (mesmo aluno aparecendo mais de uma vez)
qtd_ids_duplicados = df.duplicated(subset="Student_ID").sum()
print(f"Quantidade de Student_ID duplicados: {qtd_ids_duplicados}")

if qtd_ids_duplicados > 0:
    print("\nExemplos de registros com Student_ID duplicado:")
    print(df[df.duplicated(subset="Student_ID", keep=False)].sort_values(by="Student_ID").head(10))

# ---- Tratamento: remoção das duplicatas por Student_ID ----
df = df.drop_duplicates(subset="Student_ID", keep="first")
print(f"\nFormato do DataFrame após remoção de duplicatas por Student_ID: {df.shape}")

### Ajustes de formatação

In [ ]:
# Garante que as colunas de GPA são numéricas
df["GPA_Baseline"] = pd.to_numeric(df["GPA_Baseline"], errors="coerce")
df["GPA_Post_AI"] = pd.to_numeric(df["GPA_Post_AI"], errors="coerce")

# Garante que a frequência de uso também é numérica
df["Task_Frequency_Daily"] = pd.to_numeric(df["Task_Frequency_Daily"], errors="coerce")

# Remove linhas com dados faltando nas colunas essenciais
df = df.dropna(subset=["GPA_Baseline", "GPA_Post_AI"])

# Remove linhas totalmente vazias
df = df.dropna(how="all")